In [1]:
"""
MAP Competition - Simple DeepSeek-7B Inference
Single model inference for Kaggle submission
"""

import os
import gc
import warnings
warnings.filterwarnings('ignore')

def log(msg):
    print(msg, flush=True)

log("Starting imports...")

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
from scipy.special import softmax
from tqdm import tqdm

# ============================================================================
# CONFIGURATION
# ============================================================================

TRAIN_PATH = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/train.csv"
TEST_PATH = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/test.csv"
MODEL_PATH = "/kaggle/input/datasets/mdmarufhasanrubab/map-ds-deepseek7b"
MAX_LEN = 256
BATCH_SIZE = 4

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def find_model_path(base_path):
    """Find the actual model directory within a dataset."""
    import glob

    if not os.path.exists(base_path):
        return base_path

    # Check base path first
    if os.path.exists(os.path.join(base_path, "config.json")):
        return base_path

    # Search for config.json with model weights
    for depth in range(1, 5):
        pattern = os.path.join(base_path, "/".join(["*"] * depth), "config.json")
        matches = glob.glob(pattern)
        for m in matches:
            dir_path = os.path.dirname(m)
            has_weights = (os.path.exists(os.path.join(dir_path, "model.safetensors")) or
                          os.path.exists(os.path.join(dir_path, "pytorch_model.bin")) or
                          glob.glob(os.path.join(dir_path, "model*.safetensors")))
            if has_weights:
                return dir_path

    return base_path


def format_input(row):
    """Format input text for DeepSeek model."""
    x = "This answer is correct." if row['is_correct'] else "This answer is incorrect."
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"{x}\n"
        f"Student Explanation: {row['StudentExplanation']}"
    )


# ============================================================================
# MAIN
# ============================================================================

def main():
    log("="*60)
    log("MAP COMPETITION - DEEPSEEK-7B INFERENCE")
    log("="*60)

    if torch.cuda.is_available():
        log(f"GPU: {torch.cuda.get_device_name(0)}")

    # Load data
    log("Loading data...")
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)

    log(f"Train: {train.shape}, Test: {test.shape}")

    # Prepare labels
    train['Misconception'] = train['Misconception'].fillna('NA')
    train['target'] = train['Category'] + ':' + train['Misconception']

    le = LabelEncoder()
    le.fit(train['target'])
    n_classes = len(le.classes_)
    log(f"Number of classes: {n_classes}")

    # Find correct answers for is_correct flag
    idx = train['Category'].apply(lambda c: c.split('_')[0]) == 'True'
    correct = train.loc[idx].copy()
    correct['c'] = correct.groupby(['QuestionId', 'MC_Answer'])['MC_Answer'].transform('count')
    correct = correct.sort_values('c', ascending=False).drop_duplicates(['QuestionId'])
    correct = correct[['QuestionId', 'MC_Answer']]
    correct['is_correct'] = 1

    # Merge is_correct into test
    test = test.merge(correct, on=['QuestionId', 'MC_Answer'], how='left')
    test['is_correct'] = test['is_correct'].fillna(0)

    # Format input text
    test['text'] = test.apply(format_input, axis=1)

    # Build family map for filtering
    fam_map = {}
    for _, row in test.iterrows():
        fam_map[row['row_id']] = 'True_' if row['is_correct'] == 1 else 'False_'

    # Find and load model
    log(f"Finding model in: {MODEL_PATH}")
    model_path = find_model_path(MODEL_PATH)
    log(f"Found model at: {model_path}")

    # Load tokenizer
    log("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load model
    log("Loading model...")
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        num_labels=n_classes,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map=device
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    log("Model loaded!")

    # Tokenize
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

    ds_test = Dataset.from_pandas(test[['text']])
    ds_test = ds_test.map(tokenize, batched=True, remove_columns=['text'])

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    dataloader = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

    # Run inference
    log(f"Running inference on {len(ds_test)} samples...")
    all_logits = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Inference"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            all_logits.append(outputs.logits.float().cpu().numpy())

    predictions = np.concatenate(all_logits, axis=0)
    probs = softmax(predictions, axis=1)
    top_indices = np.argsort(-probs, axis=1)

    # Decode labels
    flat_indices = top_indices.flatten()
    decoded_labels = le.inverse_transform(flat_indices)
    top_labels = decoded_labels.reshape(top_indices.shape)

    # Apply family filter and create submission
    log("Creating submission with family filtering...")
    final_predictions = []

    for i, row_id in enumerate(test.row_id.values):
        pref = fam_map.get(row_id, 'False_')

        # Filter predictions by family prefix
        filtered = [lbl for lbl in top_labels[i] if lbl.startswith(pref)]

        # Take top 3
        top3 = filtered[:3]

        # Fill if needed
        while len(top3) < 3:
            filler = f"{pref}Neither:NA"
            if filler not in top3:
                top3.append(filler)
            else:
                top3.append(f"{pref}Correct:NA" if pref == "True_" else f"{pref}Misconception:NA")

        final_predictions.append(" ".join(top3[:3]))

    # Save submission
    submission = pd.DataFrame({
        'row_id': test.row_id.values,
        'Category:Misconception': final_predictions
    })
    submission.to_csv('submission.csv', index=False)

    log("\n" + "="*60)
    log("SUBMISSION SAVED: submission.csv")
    log("="*60)
    log(str(submission.head(10)))
    log(f"\nTotal predictions: {len(submission)}")

    # Cleanup
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


if __name__ == "__main__":
    main()

Starting imports...
MAP COMPETITION - DEEPSEEK-7B INFERENCE
GPU: Tesla T4
Loading data...
Train: (36696, 7), Test: (3, 5)
Number of classes: 65
Finding model in: /kaggle/input/datasets/mdmarufhasanrubab/map-ds-deepseek7b
Found model at: /kaggle/input/datasets/mdmarufhasanrubab/map-ds-deepseek7b/saved_model/deepseek_model
Loading tokenizer...
Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

Model loaded!


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Running inference on 3 samples...


Inference: 100%|██████████| 1/1 [00:01<00:00,  1.04s/it]

Creating submission with family filtering...

SUBMISSION SAVED: submission.csv
   row_id                             Category:Misconception
0   36696  True_Correct:NA True_Neither:NA True_Misconcep...
1   36697  False_Misconception:WNB False_Neither:NA False...
2   36698  True_Neither:NA True_Correct:NA True_Misconcep...

Total predictions: 3


## Cell 8 — Explainable AI: Misconception Knowledge Base

A key differentiator of this pipeline is **explainability**. We build a **Misconception Knowledge Base (KB)** using an LLM:

1. For each misconception type (e.g., "Additive", "Duplication"), sample representative student explanations.
2. Prompt a large language model (Qwen 2.5 7B Instruct or FLAN-T5 fallback) to generate a structured "knowledge card":
   - Plain definition
   - Typical wrong thought patterns
   - Correct rule
   - Fix steps for teachers
   - Quick self-check question
3. Cache the KB as JSON for reuse.

This enables the pipeline to not only **predict** misconceptions but also **explain** them to teachers — a critical real-world requirement.

> **Reference**: Qwen Team (2024). *Qwen2.5 Technical Report.* [arXiv:2412.15115](https://arxiv.org/abs/2412.15115)

In [ ]:
# ============================================================
# EXPLAINABLE AI — MISCONCEPTION KNOWLEDGE BASE
# ============================================================

def load_explainer():
    """
    Load the best available LLM for generating explanations.
    Priority: Qwen 2.5 7B (4-bit GPU) > FLAN-T5 Large (CPU/GPU)
    """
    if not USE_LLM_EXPLAINER:
        return None, None, None

    from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM

    # Try GPU 4-bit Qwen first
    if device == "cuda":
        try:
            from transformers import BitsAndBytesConfig
            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )
            tok = AutoTokenizer.from_pretrained(EXPLAINER_MODEL, use_fast=True)
            mdl = AutoModelForCausalLM.from_pretrained(
                EXPLAINER_MODEL,
                quantization_config=bnb_cfg,
                device_map="auto",
                torch_dtype=torch.float16,
            )
            mdl.eval()
            print(f"Loaded explainer (4-bit GPU): {EXPLAINER_MODEL}")
            return "causal", mdl, tok
        except Exception as e:
            print(f"4-bit explainer failed: {str(e)[:100]}")

    # Fallback: FLAN-T5
    tok = AutoTokenizer.from_pretrained(EXPLAINER_FALLBACK_MODEL, use_fast=True)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(EXPLAINER_FALLBACK_MODEL).to(device)
    mdl.eval()
    print(f"Loaded explainer (fallback): {EXPLAINER_FALLBACK_MODEL}")
    return "seq2seq", mdl, tok


def llm_generate(prompt: str, max_new_tokens=320, explainer_kind=None, explainer_mdl=None, explainer_tok=None):
    """Generate text from the loaded LLM."""
    if explainer_mdl is None:
        return ""
    if explainer_kind == "causal":
        inputs = explainer_tok(prompt, return_tensors="pt", truncation=True, max_length=2048).to(explainer_mdl.device)
        with torch.no_grad():
            out = explainer_mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1)
        text = explainer_tok.decode(out[0], skip_special_tokens=True)
        return text[len(prompt):].strip() if text.startswith(prompt) else text.strip()
    else:
        inputs = explainer_tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
        with torch.no_grad():
            out = explainer_mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=4)
        return explainer_tok.decode(out[0], skip_special_tokens=True).strip()


# Load explainer
EXPLAINER_KIND, explainer_model, explainer_tok = load_explainer()

# ============================================================
# BUILD MISCONCEPTION KNOWLEDGE BASE
# ============================================================

train_df["miscon_only"] = train_df["Misconception"].astype(str)
freq = Counter(train_df["miscon_only"].tolist())
top_miscons = [m for m, _ in freq.most_common(KB_BUILD_TOP_N) if m != "NA"]
print(f"\nBuilding KB for {len(top_miscons)} misconception types...")

if KB_PATH.exists():
    kb = json.loads(KB_PATH.read_text(encoding="utf-8"))
    print(f"Loaded cached KB: {len(kb)} entries")
else:
    kb = {}
    for i, miscon in enumerate(top_miscons, start=1):
        sub = train_df[train_df["miscon_only"] == miscon]
        sample = sub.sample(n=min(KB_EXAMPLES_PER_MISCON, len(sub)), random_state=42)

        examples = []
        for _, r in sample.iterrows():
            examples.append({
                "question": str(r["QuestionText"])[:200],
                "student_answer": str(r["MC_Answer"]),
                "student_expl": str(r["StudentExplanation"])[:200],
            })

        prompt = f"""You are a math teacher writing a misconception knowledge card.

Misconception name: {miscon}

Student examples:
{json.dumps(examples, ensure_ascii=False, indent=2)}

Return ONLY JSON with keys:
plain_definition (1-2 sentences),
typical_wrong_thought (list of strings),
correct_rule (list of strings),
fix_steps (list of 4-7 steps),
quick_check (one question string),
tiny_example (2-4 line string).
"""
        out = llm_generate(prompt, 420, EXPLAINER_KIND, explainer_model, explainer_tok)

        card = None
        if out:
            m = re.search(r"\{.*\}", out, flags=re.S)
            if m:
                try:
                    card = json.loads(m.group(0))
                except Exception:
                    card = None

        if card is None:
            card = {
                "plain_definition": f"Students make this mistake: {miscon}.",
                "typical_wrong_thought": ["They apply an incorrect rule or shortcut."],
                "correct_rule": ["Use the correct definition and check carefully."],
                "fix_steps": ["Restate the goal.", "Write the correct rule.", "Apply step-by-step.", "Check with an easy example."],
                "quick_check": "Can you test your rule on a simpler number?",
                "tiny_example": "Example: Compare decimals by place value, not by digit count."
            }

        kb[miscon] = card
        if i % 20 == 0:
            print(f"  Built {i}/{len(top_miscons)} cards...")

    KB_PATH.write_text(json.dumps(kb, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved KB: {len(kb)} entries to {KB_PATH}")

# Show a sample card
sample_key = next(iter(kb.keys()))
print(f"\n--- Sample KB Card: '{sample_key}' ---")
print(json.dumps(kb[sample_key], ensure_ascii=False, indent=2)[:800])

## Cell 9 — Explainable AI: Teacher Feedback Generator

The final explainability component generates **personalized teacher feedback** for each student:
- Uses the DeBERTa prediction + KB card to produce targeted feedback
- Includes: what the student seems to think, the misconception in simple words, step-by-step correct logic, and a self-check question
- Rule-based verification ensures consistency (e.g., don't say "wrong" if the student was correct)
- Optional LLM-based verification to fix contradictions

In [ ]:
# ============================================================
# EXPLAINABLE AI — TEACHER FEEDBACK GENERATOR
# ============================================================

def get_correct_choice(qid):
    """Look up the inferred correct answer for a question."""
    return str(correct_by_q_full.get(qid, ""))


def rule_based_verify(feedback: str, is_correct: int, correct_choice: str) -> str:
    """
    Rule-based post-hoc check:
    - If student is correct, soften negative language
    - If student is incorrect and we know the answer, prepend it
    """
    txt = feedback.strip()
    if is_correct == 1:
        txt = re.sub(r"\bincorrect\b", "not quite right", txt, flags=re.I)
        txt = re.sub(r"\bwrong\b", "not quite right", txt, flags=re.I)
    if is_correct == 0 and correct_choice and correct_choice not in txt:
        txt = f"Correct answer: {correct_choice}\n\n" + txt
    return txt


def llm_verify(feedback: str) -> str:
    """Optional LLM-based verification to fix contradictions."""
    if not USE_LLM_VERIFIER or explainer_model is None:
        return feedback
    prompt = f"Fix contradictions and make this concise (8-14 lines), student-friendly:\n\n{feedback}"
    out = llm_generate(prompt, 260, EXPLAINER_KIND, explainer_model, explainer_tok)
    return out if out else feedback


def generate_teacher_feedback(row, pred_label: str, pred_top3: list) -> str:
    """
    Generate an explanation for the predicted misconception.
    Combines: DeBERTa prediction + Knowledge Base card + LLM generation.
    """
    qtext   = str(row["QuestionText"])
    ans     = str(row["MC_Answer"])
    expl    = str(row["StudentExplanation"])
    qid     = row["QuestionId"]
    correct = get_correct_choice(qid)
    ic      = int(row.get("is_correct", -1))
    ic_str  = {1: "Yes", 0: "No"}.get(ic, "Unknown")

    miscon = pred_label.split(":", 1)[1] if ":" in pred_label else pred_label
    card   = kb.get(miscon, {})

    if USE_LLM_EXPLAINER and explainer_model is not None:
        prompt = f"""You are a kind, clear math teacher. Use ONLY the info below.

Question: {qtext[:300]}
Student chose: {ans}
Is correct? {ic_str}
Correct choice: {correct}
Student explanation: {expl[:300]}
Top predictions: {pred_top3}
Misconception knowledge card:
{json.dumps(card, ensure_ascii=False, indent=2)[:600]}

Write:
- 1–2 lines: what the student seems to think
- 1 line: misconception in simple words
- 4–7 short steps: correct logic
- 1 line: self-check question
Keep it concise and encouraging.
"""
        fb = llm_generate(prompt, 320, EXPLAINER_KIND, explainer_model, explainer_tok) or "Feedback generation failed."
    else:
        fb = f"""Teacher Feedback
Student chose: {ans} | Correct: {ic_str} | Correct answer: {correct}
Predicted misconception: {pred_label}

Steps to fix:
1) Re-read what the question is asking.
2) Write down the correct mathematical rule.
3) Apply it step-by-step.
4) Check with a simpler example.

Self-check: Does your rule still work if the numbers change slightly?
"""

    fb = rule_based_verify(fb, ic, correct)
    fb = llm_verify(fb)
    return fb.strip()


# ============================================================
# DEMO: Generate feedback for 5 random samples
# ============================================================

print("=" * 80)
print("  EXPLAINABLE AI — SAMPLE TEACHER FEEDBACK")
print("=" * 80)

rng = np.random.default_rng(42)
demo_idxs = rng.choice(len(train_df), size=5, replace=False)

for idx in demo_idxs:
    row = train_df.iloc[idx]
    top3 = row["pred_top3"].split()[:3] if pd.notna(row.get("pred_top3")) else []
    pred1 = top3[0] if top3 else "Unknown:Unknown"

    print(f"\n{'─'*80}")
    print(f"ROW {idx}")
    print(f"{'─'*80}")
    print(f"Question: {str(row['QuestionText'])[:150]}")
    print(f"Student Answer: {row['MC_Answer']}")
    print(f"Student Explanation: {str(row['StudentExplanation'])[:150]}")
    print(f"True Label: {row['label_str']}")
    print(f"Predictions: {top3}")

    correct_marker = "CORRECT" if (top3 and top3[0] == row["label_str"]) else "WRONG"
    print(f"Top-1 Match: {correct_marker}")

    print(f"\n--- Teacher Feedback ---")
    feedback = generate_teacher_feedback(row, pred1, top3)
    print(feedback)
    print()